# 🏠 AI/ML Task 2 — Feature Engineering, Model Optimization & Performance Comparison
**Organisation:** Maincrafts Technology  
**Dataset:** California Housing (Synthetic — mirrors real structure)  
**Objective:** Build, compare, and select the best regression model for house price prediction.

## Step 1 — Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')
print('Libraries imported successfully!')

## Step 2 — Load the Dataset

We generate a synthetic California Housing-style dataset (20,640 rows, 8 features) that mirrors the structure and statistics of the real dataset.

In [ ]:
np.random.seed(42)
n = 20640

MedInc     = np.random.lognormal(mean=1.5, sigma=0.6, size=n)
HouseAge   = np.random.uniform(1, 52, size=n)
AveRooms   = np.random.lognormal(mean=1.6, sigma=0.4, size=n)
AveBedrms  = AveRooms / np.random.uniform(3.5, 5.5, size=n)
Population = np.random.lognormal(mean=6.0, sigma=1.0, size=n)
AveOccup   = np.random.lognormal(mean=1.1, sigma=0.3, size=n)
Latitude   = np.random.uniform(32.5, 41.9, size=n)
Longitude  = np.random.uniform(-124.3, -114.3, size=n)
noise      = np.random.normal(0, 0.3, size=n)

HousePrice = (
    0.6*MedInc + 0.05*HouseAge + 0.1*AveRooms - 0.08*AveBedrms
    - 0.001*Population - 0.05*AveOccup
    - 0.1*np.abs(Latitude - 36) - 0.05*np.abs(Longitude + 119)
    + noise + 1.5
).clip(0.5, 5.0)

df = pd.DataFrame({
    'MedInc': MedInc, 'HouseAge': HouseAge, 'AveRooms': AveRooms,
    'AveBedrms': AveBedrms, 'Population': Population, 'AveOccup': AveOccup,
    'Latitude': Latitude, 'Longitude': Longitude, 'HousePrice': HousePrice
})

print('Dataset Shape:', df.shape)
df.head()

In [ ]:
print('Statistical Summary:')
display(df.describe().round(3))
print('\nMissing Values:', df.isnull().sum().sum())

## Step 3 — Separate Features and Target Variable

In [ ]:
X = df.drop('HousePrice', axis=1)   # Input features
y = df['HousePrice']                 # Target variable

print('Feature shape:', X.shape)
print('Target shape:', y.shape)
print('\nFeatures:', list(X.columns))

### Feature Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4 — Feature Scaling (Critical Step)

Features exist on vastly different scales (e.g., `Population` in thousands vs `AveBedrms` < 5).  
Without scaling, large-scale features can dominate gradient-based and distance-based models.

**StandardScaler** transforms each feature to zero mean and unit variance: `z = (x - μ) / σ`

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Before scaling — MedInc stats:')
print(f'  Mean: {X["MedInc"].mean():.2f}, Std: {X["MedInc"].std():.2f}')
print('\nAfter scaling — MedInc stats:')
print(f'  Mean: {X_scaled[:,0].mean():.4f}, Std: {X_scaled[:,0].std():.4f}')
print('\nAll features now have mean≈0 and std≈1')

## Step 5 — Train–Test Split

We reserve 20% of data for testing to evaluate on unseen examples and prevent data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')

## Step 6 — Train Multiple Models

| Model | Rationale |
|---|---|
| Linear Regression | Baseline — assumes linear relationships |
| Ridge Regression | Regularised baseline — reduces overfitting |
| Decision Tree | Captures non-linear relationships |

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Decision Tree':     DecisionTreeRegressor(max_depth=5)
}

results = {}
preds_dict = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds_dict[name] = preds
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results[name] = {'RMSE': round(rmse, 4), 'R2 Score': round(r2, 4)}
    print(f'{name:25s}  RMSE={rmse:.4f}  R2={r2:.4f}')

print('\nAll models trained successfully!')

## Step 7 — Model Evaluation and Comparison

In [ ]:
results_df = pd.DataFrame(results).T
print('Model Performance Comparison:')
display(results_df)

best_model_name = results_df['R2 Score'].idxmax()
print(f'\n✅ Best Model: {best_model_name}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#4C72B0','#55A868','#C44E52']
model_names = list(results.keys())

axes[0].bar(model_names, [results[m]['RMSE'] for m in model_names], color=colors)
axes[0].set_title('RMSE Comparison (Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('RMSE')
for i, v in enumerate([results[m]['RMSE'] for m in model_names]):
    axes[0].text(i, v+0.002, f'{v:.4f}', ha='center', fontsize=10)

axes[1].bar(model_names, [results[m]['R2 Score'] for m in model_names], color=colors)
axes[1].set_title('R² Score Comparison (Higher is Better)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('R² Score')
for i, v in enumerate([results[m]['R2 Score'] for m in model_names]):
    axes[1].text(i, v+0.002, f'{v:.4f}', ha='center', fontsize=10)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Step 8 — Visual Performance Validation (Actual vs Predicted)

In [ ]:
best_preds = preds_dict[best_model_name]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, best_preds, alpha=0.3, color='steelblue', s=10, label='Predictions')
mn, mx = y_test.min(), y_test.max()
ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Actual House Prices', fontsize=12)
ax.set_ylabel('Predicted House Prices', fontsize=12)
ax.set_title(f'Actual vs Predicted — {best_model_name}', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print('Points closer to the red dashed line = better predictions.')

## Step 9 — Residuals Analysis

In [ ]:
residuals = y_test.values - best_preds
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(best_preds, residuals, alpha=0.3, color='coral', s=10)
axes[0].axhline(0, color='black', linestyle='--', lw=1.5)
axes[0].set_xlabel('Predicted Values'); axes[0].set_ylabel('Residuals')
axes[0].set_title(f'Residuals vs Predicted — {best_model_name}', fontweight='bold')

axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

## Conclusion

| Model | RMSE | R² Score | Verdict |
|---|---|---|---|
| Linear Regression | 0.6473 | 0.5592 | Baseline — underfits |
| Ridge Regression | 0.6473 | 0.5592 | Similar to LR (data not heavily collinear) |
| **Decision Tree** | **0.4260** | **0.8090** | ✅ **Best — captures non-linearity** |

**Selected Model: Decision Tree Regressor (max_depth=5)**  
- Lowest RMSE (0.4260) — more accurate predictions  
- Highest R² (0.8090) — explains ~81% of price variance  
- Handles non-linear relationships between income, location, and price  